# 0630 과제 업데이트 버전

이 노트북은 기존 baseline을 바탕으로 아래 개선 사항을 반영한 버전입니다.

1. `unixReviewTime`을 `review_age`로 변환하여 입력 feature에 추가  
2. `helpful_yes`, `helpful_total`에 `log1p` 변환 적용  
3. 제출 전 `helpful_yes >= 0`, `helpful_total >= helpful_yes` 조건 보정  
4. validation 점수를 leaderboard와 동일한 RMSE 평균 방식으로 계산  

예상 데이터 컬럼:

- `reviewText`
- `summary`
- `unixReviewTime`
- `overall`
- `helpful` 또는 `helpful_yes`, `helpful_total`

`helpful` 컬럼이 `[도움됨 수, 전체 평가 수]` 형태라면 자동으로 `helpful_yes`, `helpful_total`로 분리합니다.


In [5]:
# 필요한 경우 아래 주석을 해제하여 설치하세요.
# %pip install transformers scikit-learn torch pandas numpy

In [6]:
import os
import ast
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from transformers import BertTokenizer

import warnings
warnings.filterwarnings("ignore")


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

device: cuda


In [7]:
# =========================
# 기본 설정
# =========================

MAX_LEN_REVIEW = 100
MAX_LEN_SUMMARY = 10

BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3

EMBED_DIM = 128
HIDDEN_DIM = 64

MODEL_NAME = "bert-base-uncased"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

VOCAB_SIZE = tokenizer.vocab_size
PAD_ID = tokenizer.pad_token_id

print("vocab size:", VOCAB_SIZE)
print("pad token id:", PAD_ID)

vocab size: 30522
pad token id: 0


## 1. 데이터 불러오기

아래 코드는 다음 순서로 데이터를 찾습니다.

1. 현재 노트북에 이미 `df` 변수가 있으면 그것을 학습 데이터로 사용  
2. 없으면 `train.csv`를 읽어서 학습 데이터로 사용  
3. test 데이터는 `test.csv`가 있으면 자동으로 읽음  
4. `sample_submission.csv`가 있으면 제출 형식을 그대로 사용  


In [ ]:
# =========================
# 데이터 로드
# =========================

TRAIN_PATH = "train.csv"
TEST_PATH = "test.csv"
SAMPLE_SUBMISSION_PATH = "sample_submission.csv"

try:
    train_path = "train.jsonl"
    df = pd.read_json(train_path, lines=True)
except NameError:
    if os.path.exists(TRAIN_PATH):
        df = pd.read_csv(TRAIN_PATH)
        print(f"{TRAIN_PATH} 파일을 학습 데이터로 불러왔습니다.")
    else:
        raise FileNotFoundError(
            "학습 데이터가 없습니다. df 변수를 미리 만들거나 train.csv 파일을 같은 폴더에 두세요."
        )

if os.path.exists(TEST_PATH):
    test_df = pd.read_csv(TEST_PATH)
    print(f"{TEST_PATH} 파일을 test 데이터로 불러왔습니다.")
else:
    test_df = None
    print("test.csv 파일이 없어 test 예측 부분은 나중에 실행하세요.")

if os.path.exists(SAMPLE_SUBMISSION_PATH):
    sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
    print(f"{SAMPLE_SUBMISSION_PATH} 파일을 불러왔습니다.")
else:
    sample_submission = None
    print("sample_submission.csv 파일이 없습니다. 기본 submission 형식으로 생성합니다.")

print("train shape:", df.shape)
display(df.head())

기존 df 변수를 학습 데이터로 사용합니다.
test.csv 파일이 없어 test 예측 부분은 나중에 실행하세요.
sample_submission.csv 파일이 없습니다. 기본 submission 형식으로 생성합니다.
train shape: (9234, 10)


,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,row_id
0,A1K582XYLTKUCD,B0042F1L4S,"Coder10 ""Ricardo""","[0, 1]",im not a profesional player but i like to play...,5,Everything a hobbiest want,1361404800,"02 21, 2013",8492
1,A2J4UAF6RW13WK,B000EELB8W,Michael W DeSilva,"[0, 0]",This is just an excellent product. I have been...,5,Excellent Product,1371686400,"06 20, 2013",4666
2,A3OXHLG6DIBRW8,B000BU5V58,"C. Hill ""CFH""","[1, 1]","We opted for the World Tour ""Guitar Gig Bag"" o...",4,Roomy Guitar Case - Recommended,1290556800,"11 24, 2010",4286
3,A3872Y2XH0YDX1,B000CZ0RLK,Amazon Customer,"[0, 0]","This is not a top tier condenser mic, but it i...",5,definitely a good value,1311552000,"07 25, 2011",4446
4,A2G3VQU2GRN8BU,B000978D58,"Moral Hazard ""D""","[1, 3]",This could be used for lightweight microphones...,3,Too cheap.,1316044800,"09 15, 2011",3921


## 2. helpful 컬럼 분리 및 기본 전처리

`helpful` 컬럼이 `[3, 5]` 또는 `"[3, 5]"`처럼 들어 있다면 다음 코드가 자동으로 분리합니다.

- `helpful_yes`: 도움이 됐다고 평가한 수
- `helpful_total`: 도움 여부를 평가한 전체 수


In [ ]:
def parse_helpful(x):
    if isinstance(x, list) and len(x) == 2:
        return x[0], x[1]

    if isinstance(x, str):
        try:
            y = ast.literal_eval(x)
            if isinstance(y, (list, tuple)) and len(y) == 2:
                return y[0], y[1]
        except Exception:
            return 0, 0

    return 0, 0


def prepare_helpful_columns(data):
    data = data.copy()
    if ("helpful_yes" not in data.columns or "helpful_total" not in data.columns) and "helpful" in data.columns:
        data["helpful_yes"], data["helpful_total"] = zip(*data["helpful"].apply(parse_helpful))

    if "helpful_yes" in data.columns:
        data["helpful_yes"] = pd.to_numeric(data["helpful_yes"], errors="coerce").fillna(0)
    if "helpful_total" in data.columns:
        data["helpful_total"] = pd.to_numeric(data["helpful_total"], errors="coerce").fillna(0)

    return data


df = prepare_helpful_columns(df)

required_train_cols = ["reviewText", "summary", "unixReviewTime", "overall", "helpful_yes", "helpful_total"]
missing_cols = [col for col in required_train_cols if col not in df.columns]

if missing_cols:
    raise ValueError(f"학습 데이터에 필요한 컬럼이 없습니다: {missing_cols}")

print(df[required_train_cols].head())

                                          reviewText  \
0  im not a profesional player but i like to play...   
1  This is just an excellent product. I have been...   
2  We opted for the World Tour "Guitar Gig Bag" o...   
3  This is not a top tier condenser mic, but it i...   
4  This could be used for lightweight microphones...   

                           summary  unixReviewTime  overall  helpful_yes  \
0       Everything a hobbiest want      1361404800        5            0   
1                Excellent Product      1371686400        5            0   
2  Roomy Guitar Case - Recommended      1290556800        4            1   
3          definitely a good value      1311552000        5            0   
4                       Too cheap.      1316044800        3            1   

   helpful_total  
0              1  
1              0  
2              1  
3              0  
4              3  


## 3. feature 생성

핵심 수정 사항 1번입니다.

`unixReviewTime`을 그대로 넣지 않고, 학습 데이터 기준 가장 최근 시간에서 현재 리뷰 시간을 뺀 `review_age`를 만듭니다.

- 오래된 리뷰일수록 `review_age_days`가 큼
- 최근 리뷰일수록 `review_age_days`가 작음

helpful 수는 오래 노출된 리뷰일수록 많이 쌓일 가능성이 있으므로 `review_age_days`가 중요한 feature가 될 수 있습니다.


In [ ]:
def make_features(data, reference_time):
    data = data.copy()

    data["reviewText"] = data["reviewText"].fillna("")
    data["summary"] = data["summary"].fillna("")
    data["unixReviewTime"] = pd.to_numeric(data["unixReviewTime"], errors="coerce").fillna(reference_time)

    data["len"] = data["reviewText"].apply(lambda x: len(str(x).split()))
    data["summary_len"] = data["summary"].apply(lambda x: len(str(x).split()))

    # 1. unixReviewTime -> review_age 변환 (review_age가 클수록 오래된 리뷰)
    data["review_age"] = reference_time - data["unixReviewTime"]
    data["review_age_days"] = data["review_age"] / (60 * 60 * 24)

    return data


def make_targets(data):
    data = data.copy()

    # 2. helpful_yes, helpful_total에 log1p 적용(0 비율이 높아서)
    data["helpful_yes_log"] = np.log1p(data["helpful_yes"].clip(lower=0))
    data["helpful_total_log"] = np.log1p(data["helpful_total"].clip(lower=0))

    return data

In [11]:
# =========================
# Train / Validation 분리
# =========================

train_raw, valid_raw = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

train_raw = train_raw.reset_index(drop=True)
valid_raw = valid_raw.reset_index(drop=True)

# train 기준으로 reference_time 설정
REFERENCE_TIME = train_raw["unixReviewTime"].max()

train_df = make_features(train_raw, REFERENCE_TIME)
valid_df = make_features(valid_raw, REFERENCE_TIME)

train_df = make_targets(train_df)
valid_df = make_targets(valid_df)

print("train_df:", train_df.shape)
print("valid_df:", valid_df.shape)

display(train_df[["unixReviewTime", "review_age_days", "len", "summary_len", "overall", "helpful_yes", "helpful_total", "helpful_yes_log", "helpful_total_log"]].head())

train_df: (7387, 18)
valid_df: (1847, 18)


,unixReviewTime,review_age_days,len,summary_len,overall,helpful_yes,helpful_total,helpful_yes_log,helpful_total_log
0,1387497600,214.0,71,2,4,0,0,0.000000,0.000000
1,1353542400,607.0,33,7,2,0,1,0.000000,0.693147
2,1392854400,152.0,38,5,5,0,0,0.000000,0.000000
3,1317600000,1023.0,35,1,5,0,0,0.000000,0.000000
4,1282694400,1427.0,281,7,5,32,33,3.496508,3.526361


In [ ]:
# 수치형 feature scaling
numeric_cols = ["review_age_days", "len", "summary_len"]

scaler = StandardScaler()

train_df[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
valid_df[numeric_cols] = scaler.transform(valid_df[numeric_cols])

display(train_df[numeric_cols].head())

,review_age_days,len,summary_len
0,-0.717296,-0.180165,-0.831145
1,0.167049,-0.531083,0.932722
2,-0.856811,-0.484909,0.227175
3,1.103150,-0.512613,-1.183918
4,2.012248,1.759114,0.932722


## 4. Dataset 구성

리뷰 본문과 summary를 각각 토큰화합니다.

`attention_mask`를 함께 저장하여 LSTM이 padding 토큰을 무시할 수 있도록 합니다.


In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, data, tokenizer, max_len_review, max_len_summary, is_train=True):
        self.data = data.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len_review = max_len_review
        self.max_len_summary = max_len_summary
        self.is_train = is_train

    def __len__(self):
        return len(self.data)

    # 텍스트를 토크나이징하고 인코딩
    def encode_text(self, text, max_len):
        encoded = self.tokenizer(
            str(text),
            padding="max_length",
            truncation=True,
            max_length=max_len,
            return_attention_mask=True,
            return_tensors="pt"
        )

        input_ids = encoded["input_ids"].squeeze(0)
        attention_mask = encoded["attention_mask"].squeeze(0)

        return input_ids, attention_mask

    def __getitem__(self, idx):
        row = self.data.iloc[idx]

        review_ids, review_mask = self.encode_text(row["reviewText"], self.max_len_review)
        summary_ids, summary_mask = self.encode_text(row["summary"], self.max_len_summary)

        numeric_features = torch.tensor(
            row[numeric_cols].values.astype(np.float32),
            dtype=torch.float32
        )

        item = {
            "review_ids": review_ids,
            "review_mask": review_mask,
            "summary_ids": summary_ids,
            "summary_mask": summary_mask,
            "numeric_features": numeric_features
        }

        if self.is_train:
            # 학습 target은 overall + helpful log값
            labels = torch.tensor(
                [
                    row["overall"],
                    row["helpful_yes_log"],
                    row["helpful_total_log"]
                ],
                dtype=torch.float32
            )

            # validation 점수 계산용 원래 target
            original_labels = torch.tensor(
                [
                    row["overall"],
                    row["helpful_yes"],
                    row["helpful_total"]
                ],
                dtype=torch.float32
            )

            item["labels"] = labels
            item["original_labels"] = original_labels

        return item

In [14]:
train_dataset = ReviewDataset(
    train_df,
    tokenizer,
    MAX_LEN_REVIEW,
    MAX_LEN_SUMMARY,
    is_train=True
)

valid_dataset = ReviewDataset(
    valid_df,
    tokenizer,
    MAX_LEN_REVIEW,
    MAX_LEN_SUMMARY,
    is_train=True
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("train batches:", len(train_loader))
print("valid batches:", len(valid_loader))

train batches: 462
valid batches: 116


## 5. 모델 정의

구조는 다음과 같습니다.

```text
reviewText  → Embedding → LSTM → review vector
summary     → Embedding → LSTM → summary vector
numeric features(review_age_days, len, summary_len)
        ↓
concat
        ↓
MLP regressor
        ↓
overall, helpful_yes_log, helpful_total_log
```


현재 모델은 reviewText와 summary를 각각 BERT tokenizer로 토큰화한 뒤 Embedding과 LSTM으로 문장 벡터를 만들고, 여기에 review_age_days, len, summary_len 수치형 feature를 결합하여 overall, helpful_yes_log, helpful_total_log를 동시에 예측하는 LSTM 기반 multi-input, multi-target 회귀 모델

In [15]:
class ImprovedRNNRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_numeric_features, padding_idx):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embed_dim,
            padding_idx=padding_idx
        )

        self.review_lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        self.summary_lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True
        )

        combined_dim = hidden_dim * 2 + num_numeric_features

        self.regressor = nn.Sequential(
            nn.Linear(combined_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 3)
        )

    def encode_with_lstm(self, input_ids, attention_mask, lstm):
        embedded = self.embedding(input_ids)

        lengths = attention_mask.sum(dim=1).detach().cpu()
        lengths = torch.clamp(lengths, min=1)

        packed = pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = lstm(packed)

        last_hidden = hidden[-1]

        return last_hidden

    def forward(self, review_ids, review_mask, summary_ids, summary_mask, numeric_features):
        review_vec = self.encode_with_lstm(
            review_ids,
            review_mask,
            self.review_lstm
        )

        summary_vec = self.encode_with_lstm(
            summary_ids,
            summary_mask,
            self.summary_lstm
        )

        combined = torch.cat(
            [review_vec, summary_vec, numeric_features],
            dim=1
        )

        outputs = self.regressor(combined)

        return outputs

In [16]:
model = ImprovedRNNRegressor(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_numeric_features=len(numeric_cols),
    padding_idx=PAD_ID
).to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print(model)

ImprovedRNNRegressor(
  (embedding): Embedding(30522, 128, padding_idx=0)
  (review_lstm): LSTM(128, 64, batch_first=True)
  (summary_lstm): LSTM(128, 64, batch_first=True)
  (regressor): Sequential(
    (0): Linear(in_features=131, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=3, bias=True)
  )
)


## 6. 후처리 및 leaderboard 방식 validation 점수 계산

핵심 수정 사항 3, 4번입니다.

모델은 helpful 값을 log scale로 예측하므로, validation과 submission에서는 `expm1`로 원래 스케일로 복원합니다.

그 후 제출 전 조건을 보정합니다.

- `overall`: 1 이상 5 이하
- `helpful_yes`: 0 이상
- `helpful_total`: `helpful_yes` 이상


In [17]:
def postprocess_predictions(preds):
    preds = np.asarray(preds)

    overall_pred = preds[:, 0]

    # log1p로 학습한 helpful 값을 원래 스케일로 복원
    helpful_yes_pred = np.expm1(preds[:, 1])
    helpful_total_pred = np.expm1(preds[:, 2])

    # 3. 제출 전 보정
    overall_pred = np.clip(overall_pred, 1, 5)

    helpful_yes_pred = np.clip(helpful_yes_pred, 0, None)
    helpful_total_pred = np.clip(helpful_total_pred, 0, None)

    # helpful_total은 helpful_yes보다 작을 수 없음
    helpful_total_pred = np.maximum(helpful_total_pred, helpful_yes_pred)

    final_preds = np.column_stack(
        [overall_pred, helpful_yes_pred, helpful_total_pred]
    )

    return final_preds


def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def leaderboard_score(y_true, y_pred):
    overall_rmse = rmse(y_true[:, 0], y_pred[:, 0])
    helpful_yes_rmse = rmse(y_true[:, 1], y_pred[:, 1])
    helpful_total_rmse = rmse(y_true[:, 2], y_pred[:, 2])

    final_score = (
        overall_rmse
        + helpful_yes_rmse
        + helpful_total_rmse
    ) / 3

    return {
        "Final": final_score,
        "overall_RMSE": overall_rmse,
        "helpful_yes_RMSE": helpful_yes_rmse,
        "helpful_total_RMSE": helpful_total_rmse
    }

## 7. 학습 및 검증

validation 점수는 leaderboard와 동일하게 다음 세 RMSE의 평균으로 계산합니다.

```text
Final = (overall RMSE + helpful_yes RMSE + helpful_total RMSE) / 3
```


In [18]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0

    for batch in train_loader:
        review_ids = batch["review_ids"].to(device)
        review_mask = batch["review_mask"].to(device)
        summary_ids = batch["summary_ids"].to(device)
        summary_mask = batch["summary_mask"].to(device)
        numeric_features = batch["numeric_features"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            review_ids,
            review_mask,
            summary_ids,
            summary_mask,
            numeric_features
        )

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    return avg_loss


def evaluate(model, valid_loader, criterion, device):
    model.eval()

    total_loss = 0.0

    all_preds = []
    all_true = []

    with torch.no_grad():
        for batch in valid_loader:
            review_ids = batch["review_ids"].to(device)
            review_mask = batch["review_mask"].to(device)
            summary_ids = batch["summary_ids"].to(device)
            summary_mask = batch["summary_mask"].to(device)
            numeric_features = batch["numeric_features"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                review_ids,
                review_mask,
                summary_ids,
                summary_mask,
                numeric_features
            )

            loss = criterion(outputs, labels)
            total_loss += loss.item()

            all_preds.append(outputs.detach().cpu().numpy())
            all_true.append(batch["original_labels"].numpy())

    avg_loss = total_loss / len(valid_loader)

    all_preds = np.vstack(all_preds)
    all_true = np.vstack(all_true)

    processed_preds = postprocess_predictions(all_preds)
    scores = leaderboard_score(all_true, processed_preds)

    return avg_loss, scores

In [19]:
best_score = float("inf")
best_model_path = "best_model_review_age_log1p.pt"

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    valid_loss, scores = evaluate(
        model,
        valid_loader,
        criterion,
        device
    )

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.6f}")
    print(f"Valid Loss: {valid_loss:.6f}")
    print(f"Final: {scores['Final']:.6f}")
    print(f"overall RMSE: {scores['overall_RMSE']:.6f}")
    print(f"helpful_yes RMSE: {scores['helpful_yes_RMSE']:.6f}")
    print(f"helpful_total RMSE: {scores['helpful_total_RMSE']:.6f}")

    if scores["Final"] < best_score:
        best_score = scores["Final"]
        torch.save(model.state_dict(), best_model_path)
        print("Best model saved.")

print("\nBest validation score:", best_score)


Epoch 1/10
Train Loss: 0.752556
Valid Loss: 0.423445
Final: 3.782611
overall RMSE: 0.807588
helpful_yes RMSE: 5.086760
helpful_total RMSE: 5.453486
Best model saved.

Epoch 2/10
Train Loss: 0.478037
Valid Loss: 0.459447
Final: 3.733825
overall RMSE: 0.816124
helpful_yes RMSE: 4.970286
helpful_total RMSE: 5.415065
Best model saved.

Epoch 3/10
Train Loss: 0.409243
Valid Loss: 0.435589
Final: 3.819508
overall RMSE: 0.811725
helpful_yes RMSE: 5.168186
helpful_total RMSE: 5.478612

Epoch 4/10
Train Loss: 0.329453
Valid Loss: 0.433303
Final: 3.954201
overall RMSE: 0.781894
helpful_yes RMSE: 5.235567
helpful_total RMSE: 5.845143

Epoch 5/10
Train Loss: 0.252005
Valid Loss: 0.452791
Final: 4.119754
overall RMSE: 0.794249
helpful_yes RMSE: 5.427080
helpful_total RMSE: 6.137934

Epoch 6/10
Train Loss: 0.201823
Valid Loss: 0.472783
Final: 3.921720
overall RMSE: 0.793850
helpful_yes RMSE: 5.178115
helpful_total RMSE: 5.793193

Epoch 7/10
Train Loss: 0.174784
Valid Loss: 0.457006
Final: 3.821167


## 8. Test 예측 및 submission.csv 생성

`test.csv`가 있는 경우 아래 셀을 실행하면 `submission.csv`가 생성됩니다.


In [25]:
test_df = pd.read_json("test.jsonl", lines=True)
if test_df is not None:
    test_df = make_features(test_df, REFERENCE_TIME)

    # train에서 fit한 scaler를 그대로 사용
    test_df[numeric_cols] = scaler.transform(test_df[numeric_cols])

    test_dataset = ReviewDataset(
        test_df,
        tokenizer,
        MAX_LEN_REVIEW,
        MAX_LEN_SUMMARY,
        is_train=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    print("test shape:", test_df.shape)
    print("test batches:", len(test_loader))
else:
    print("test_df가 없습니다. test.csv 파일을 준비한 뒤 이 셀부터 다시 실행하세요.")

test shape: (1027, 12)
test batches: 65


In [26]:
def predict_test(model, test_loader, device):
    model.eval()

    all_preds = []

    with torch.no_grad():
        for batch in test_loader:
            review_ids = batch["review_ids"].to(device)
            review_mask = batch["review_mask"].to(device)
            summary_ids = batch["summary_ids"].to(device)
            summary_mask = batch["summary_mask"].to(device)
            numeric_features = batch["numeric_features"].to(device)

            outputs = model(
                review_ids,
                review_mask,
                summary_ids,
                summary_mask,
                numeric_features
            )

            all_preds.append(outputs.detach().cpu().numpy())

    all_preds = np.vstack(all_preds)

    processed_preds = postprocess_predictions(all_preds)

    return processed_preds

In [27]:
if test_df is not None:
    model.load_state_dict(torch.load(best_model_path, map_location=device))

    test_preds = predict_test(
        model,
        test_loader,
        device
    )

    print(test_preds[:5])
else:
    test_preds = None
    print("test 예측을 건너뜁니다.")

[[4.6562533  0.05408221 0.06258586]
 [4.5096436  0.8501378  1.0016276 ]
 [5.         0.         0.        ]
 [5.         0.20698512 0.24475236]
 [4.9912934  0.05780041 0.05780041]]


In [28]:
if test_preds is not None:
    if sample_submission is not None:
        submission = sample_submission.copy()
    else:
        submission = pd.DataFrame(index=range(len(test_df)))

    submission["overall"] = test_preds[:, 0]
    submission["helpful_yes"] = test_preds[:, 1]
    submission["helpful_total"] = test_preds[:, 2]

    # 실수 제출이 허용되는 경우 RMSE에서는 round하지 않는 편이 유리할 수 있습니다.
    # 정수 제출이 필요하다면 아래 두 줄의 주석을 해제하세요.
    # submission["helpful_yes"] = submission["helpful_yes"].round()
    # submission["helpful_total"] = submission["helpful_total"].round()

    submission.to_csv("submission.csv", index=False)

    print("submission.csv 저장 완료")
    display(submission.head())
else:
    print("test_preds가 없어 submission.csv를 생성하지 않았습니다.")

submission.csv 저장 완료


,overall,helpful_yes,helpful_total
0,4.656253,0.054082,0.062586
1,4.509644,0.850138,1.001628
2,5.000000,0.000000,0.000000
3,5.000000,0.206985,0.244752
4,4.991293,0.057800,0.057800


## 제출 전 체크 포인트

아래 조건을 확인하세요.

- `overall` 값이 1~5 범위 안에 있는지
- `helpful_yes`가 0 이상인지
- `helpful_total`이 `helpful_yes`보다 크거나 같은지
- `sample_submission.csv`의 컬럼 구조와 최종 `submission.csv`의 컬럼 구조가 같은지


In [29]:
if "submission" in globals():
    print("overall min/max:", submission["overall"].min(), submission["overall"].max())
    print("helpful_yes min:", submission["helpful_yes"].min())
    print("helpful_total min:", submission["helpful_total"].min())
    print("helpful_total >= helpful_yes 비율:", (submission["helpful_total"] >= submission["helpful_yes"]).mean())
    print("submission shape:", submission.shape)
    display(submission.head())
else:
    print("아직 submission 변수가 없습니다.")

overall min/max: 2.5241081714630127 5.0
helpful_yes min: 0.0
helpful_total min: 0.0
helpful_total >= helpful_yes 비율: 1.0
submission shape: (1027, 3)


,overall,helpful_yes,helpful_total
0,4.656253,0.054082,0.062586
1,4.509644,0.850138,1.001628
2,5.000000,0.000000,0.000000
3,5.000000,0.206985,0.244752
4,4.991293,0.057800,0.057800
